In [ ]:
import pandas as pd

sentence_level_df = pd.read_csv(".csv")

In [ ]:
import pandas as pd

def is_dialogue(sentence):
    if isinstance(sentence, str):
        return int(any(q in sentence for q in ['"', '“', '”', '‘', '’']))
    return 0

# Apply to your DataFrame
sentence_level_df['is_dialogue'] = sentence_level_df['subject_sentences'].apply(is_dialogue)
sentence_level_df

In [ ]:
# Load pipeline
import stanza

# Load the Indonesian model once
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,lemma')

# Fungsi untuk membaca file lexicon
def load_afinn(afinn_path):
    lexicon = {}
    for path in [afinn_path]:  # Tidak perlu pakai multiplier lagi
        with open(path, encoding='utf-8') as f:
            lines = f.readlines()[1:]  # skip header
            for line in lines:
                if line.strip():
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        word = parts[0]
                        try:
                            score = float(parts[1])
                            lexicon[word] =  score
                        except ValueError:
                            continue  # skip if score is not a float
    return lexicon

def verb_adj_sentiment_stanza(sentence, lexicon):
    doc = nlp(sentence)
    verbs = []
    adjs = []

    for sent in doc.sentences:
        for word in sent.words:
            lemma = word.lemma.lower()
            if word.upos == 'VERB':
                verbs.append(lemma)
            elif word.upos == 'ADJ':
                adjs.append(lemma)

    relevant_words = verbs + adjs
    pos_count = sum(1 for word in relevant_words if lexicon.get(word, 0) > 0)
    neg_count = sum(1 for word in relevant_words if lexicon.get(word, 0) < 0)

    return pos_count - neg_count

afinn_path = "AFINN.tsv"
lexicon = load_afinn(afinn_path)

# Terapkan ke semua baris dan kembalikan dua kolom baru
sentence_level_df['count_lexicon'] = sentence_level_df['subject_sentences'].apply(
    lambda x: verb_adj_sentiment_stanza(x, lexicon)
)

In [ ]:
import stanza
import pandas as pd

# Load Indonesian NLP pipeline
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse', use_gpu=False)

def get_alias_roles(row):
    sentence = row['subject_sentences']
    aliases = row['aliases']

    is_subject = 0
    is_object = 0

    doc = nlp(sentence)

    for sent in doc.sentences:
        for word in sent.words:
            for alias in aliases:
                # Match alias exactly or by lemma
                if alias.lower() in word.text.lower():
                    # Check dependency relation
                    if word.deprel in ["nsubj", "nsubj:pass"]:
                        is_subject = 1
                    elif word.deprel in ["obj", "iobj", "obl"]:
                        is_object = 1
    return pd.Series({'is_subject': is_subject, 'is_object': is_object})

sentence_level_df[['is_subject', 'is_object']] = sentence_level_df.apply(get_alias_roles, axis=1)


In [ ]:
import pandas as pd

# Step 1: Identify the least frequent label
label_counts = sentence_level_df['type'].value_counts()
least_common_label = label_counts.idxmin()

# Step 2: Sort so that the least common label comes first (we want to keep these)
sentence_level_df_sorted = sentence_level_df.sort_values(
    by='type',
    key=lambda x: x.apply(lambda val: 0 if val == least_common_label else 1)
)

# Step 3: Drop duplicates in 'subject_sentences' keeping the first occurrence (which is from least_common_label)
sentence_level_df_dedup = sentence_level_df_sorted.drop_duplicates(subset='subject_sentences', keep='first')


In [ ]:
import re
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Stopwords bahasa Inggris
ENGLISH_STOPWORDS = set(stopwords.words('english'))

# Stemmer bahasa Inggris
stemmer = PorterStemmer()

def preprocess_sentence(sentence):
    # Handle missing value
    if pd.isna(sentence):
        return ""

    # Lowercase
    sentence = sentence.lower()

    # Tokenize: ambil kata alphabet saja
    tokens = re.findall(r'\b[a-zA-Z]+\b', sentence)

    # Remove stopwords
    tokens = [token for token in tokens if token not in ENGLISH_STOPWORDS]

    # Stem each token
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(stemmed_tokens)

sentence_level_df["subject_sentences_stem"] = sentence_level_df["subject_sentences"].apply(preprocess_sentence)

In [ ]:
# Split train-test
train_df, test_df = train_test_split(
    sentence_level_df_dedup,
    test_size=0.2,
    stratify=sentence_level_df_dedup['type'],
    random_state=42
)

# Pilih fitur dan label
X_train_exploded = train_df.copy()
y_train_exploded = train_df['type']

X_test_exploded = test_df.copy()
y_test_exploded = test_df['type']


In [ ]:
train_df['type'].value_counts()

In [ ]:
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Function to get the most common element from a list
def get_most_common_label(label_list):
    # Flatten the list if it contains nested lists
    # flat_list = [item for sublist in label_list for item in sublist]
    most_common = Counter(label_list).most_common(1)[0][0]
    return most_common

def evaluation_metric_val(best_model, val_df):
    data_test = val_df.copy()
    predictions = best_model.predict(data_test)
    data_test['predicted_type'] = predictions

    # Join alias lists into a string
    data_test["aliases"] = data_test["aliases"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

    # Group by entity and aggregate predictions and true labels
    data_test = data_test.groupby(["aliases", "story_id", "person"])[["predicted_type", "type"]].agg(list).reset_index()

    # Get most common label per group
    data_test['predicted_type_group'] = data_test['predicted_type'].apply(get_most_common_label)
    data_test['type_grouped'] = data_test['type'].apply(get_most_common_label)

    # Ground truth and predictions
    y_true = data_test['type_grouped']
    y_pred = data_test['predicted_type_group']

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Display confusion matrix
    print("Confusion Matrix:\n", cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')

    # Print classification report
    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Compute and print individual metrics
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    return data_test


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Preprocessing pipeline
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('integer', MinMaxScaler(), ["is_dialogue", "is_subject", "count_lexicon"]),
        ('text', TfidfVectorizer(), 'subject_sentences_stem')
    ]
)
# Pipeline with LinearSVC
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC())
])

# Grid search parameters
param_grid = {
    'preprocessor__text__max_df': [0.75, 1.0],
    'preprocessor__text__ngram_range': [(1, 1), (1, 2)],
    'preprocessor__text__max_features': [100, 500, 1000, 1500, 2000, 2500, 3000, 5000],
    'classifier__C': [0.1, 1, 10],  # Regularization parameter for SVM
    'classifier__loss': ['hinge', 'squared_hinge'],  # SVM loss functions
    'classifier__max_iter': [1000, 5000],  # max iterations
    'classifier__class_weight': [None, 'balanced']
}

# GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)

# Fit grid search
grid_search.fit(X_train_exploded, y_train_exploded)

# Best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

# Evaluate on test set
test_score = grid_search.score(X_test_exploded, y_test_exploded)
y_pred = grid_search.best_estimator_.predict(X_test_exploded)

print("Test set score:", test_score)
report = classification_report(y_test_exploded, y_pred)
print("Classification Report for Best Model:\n", report)

best_model_one_svm_exp_tf = grid_search.best_estimator_
train_accuracy = best_model_one_svm_exp_tf.score(X_train_exploded, y_train_exploded)
test_accuracy = best_model_one_svm_exp_tf.score(X_test_exploded, y_test_exploded)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

